In [2]:
from konlpy.tag import Mecab
import os
import json


dicpath = "/home/wagyu0923/miniconda3/envs/exaone/lib/mecab/dic/mecab-ko-dic"
m = Mecab(dicpath=dicpath)

files = os.listdir('/home/wagyu0923/project/Document_Analyzer/chunk_files')

for file in files:
    seperated_file = []
    with open (f'/home/wagyu0923/project/Document_Analyzer/chunk_files/{file}','r') as f:
        chunk_file = json.load(f)
        
    chunk_text = [text['chunk'] for text in chunk_file]
    for idx, i in enumerate(chunk_file):
        seperated_text = m.morphs(chunk_text[idx])
        i['seperated_text'] = seperated_text
        seperated_file.append(i)
    
    file_name = os.path.basename(file)
    file_name = os.path.splitext(file_name)[0]
    
    chunk_path = f'/home/wagyu0923/project/Document_Analyzer/seperated_files/{file_name}_seperated.json'

    with open(chunk_path, 'w', encoding='utf-8') as f:
        json.dump(seperated_file, f,ensure_ascii=False,indent=4)
        

In [2]:
from rank_bm25 import BM25Okapi
import os
from tkinter import filedialog

file = filedialog.askopenfilename()

import os
import json

with open(f'{file}', 'r') as f:
    seperated_file = json.load(f)

seperated_text = []
for i in seperated_file:
    seperated_text.append(i['seperated_text'])
    
bm25 = BM25Okapi(seperated_text)

In [4]:
from konlpy.tag import Mecab
dicpath = "/home/wagyu0923/miniconda3/envs/exaone/lib/mecab/dic/mecab-ko-dic"
m = Mecab(dicpath=dicpath)

query = '재무제표 분석'
seperated_query = m.morphs(query)

doc_scores = bm25.get_scores(seperated_query)
top_n_docs = bm25.get_top_n(seperated_query, seperated_file, n=7)

retrived_data = ''

for doc in top_n_docs:
    page = doc['page']
    chunk = doc['chunk']
    retrived_data += f'page : {page}, \n content : {chunk} \n\n'


In [ ]:
import ollama

OLLAMA_MODEL = 'gpt-oss:20b'
SYSTEM_PROMPT =  f"""You are an AI assistant that analyzes financial documents and outputs the results in a specific JSON format.

**Instructions:**
1. Analyze the provided CONTEXT to answer the user's QUESTION.
2. Your response **MUST BE** a single, valid JSON object.
3. Do not add any explanatory text before or after the JSON.
4. The JSON object must follow this exact structure:

**JSON Structure Example:**
```json
{{
  "risks": [
    {{
      "summary": "A brief, one-sentence summary of a single investment risk.",
      "evidence": [
        {{
          "source": "The name of the source file.",
          "page": "The page number as an integer.",
          "quote": "The exact sentence from the document that supports the summary."
        }}
      ]
    }}
  ]
}}
"""
response = ollama.chat(
    model=OLLAMA_MODEL,
    messages=[
        {'role': 'user', 'content': SYSTEM_PROMPT}
    ],
    options={

        'temperature': 0.0,
        'repeat_penalty': 1.1,
        'seed' : 42
    }
)
outputs = response['message']['content']

In [6]:
print(outputs)

**재무제표 분석**

1. **재무제표 구성 및 작성 기준**  
   - 본 보고서는 **연결재무제표**와 **반기연결재무제표**를 포함하고 있으며,  
     - **연결재무제표**는 4‑1 재무상태표, 4‑2 포괄손익계산서, 4‑3 자본변동표, 4‑4 현금흐름표, 5 재무제표 주석, 6 배당에 관한 사항, 7 증권의 발행을 통한 자금조달에 관한 사항 등으로 구성됩니다. (Source: page 1)  
   - **반기연결재무제표**는 기업회계기준서 제1034호 ‘중간재무보고’에 따라 작성되었으며, 연차재무제표에 비해 포함되는 정보가 적습니다. (Source: page 79)  
   - 반기재무제표와 별도재무제표는 각각 2024년 12월 31일 종료 회계연도 연차재무제표와 별도재무제표를 함께 활용해 이해해야 합니다. (Source: page 129)

2. **대손충당금 및 매출채권 관리**  
   - 회사는 매출채권 및 미수금 잔액에 대해 개별분석과 과거 대손경험을 토대로 예상되는 대손추정액을 대손충당금으로 설정합니다. (Source: page 184)  
   - 대손충당금 변동 현황과 설정 방침이 재무제표 주석에 상세히 기재되어 있어, 채권 회수 가능성에 대한 관리 체계를 확인할 수 있습니다. (Source: page 184)

3. **재고자산 현황**  
   - 2023년 사업연도 재고자산의 사업부문별 보유현황 합계는 **20,290,751**(단위는 문서에 명시되지 않음)이며, 이전 기간 대비 **14,128,759**로 보고됩니다. (Source: page 184)  
   - 재고자산 규모가 감소한 점은 재고 회전율 개선 또는 매출 구조 변화에 따른 재고 관리 효율성을 시사합니다.

4. **유동성 위험 관리**  
   - 연결회사는 단기 및 중장기 자금관리계획을 수립하고, 현금유출 예산과 실제 현금유출액을 지속적으로 분석·검토하여 금융부채와 금융자산의 만기구조를 대응시키고 있습니다. (Source: page 82, page 63